# 2.2 낙관적 초기화와 UCB: 불확실성을 이용한 탐험 — 실습 노트북

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SuminHan/book-ml/blob/main/notebooks/ml2/chapter02_2_ucb_optimistic_init.ipynb)

책 본문: [2.2 낙관적 초기화와 UCB](https://smhanlab.com/book-ml/kor/ml2/chapter02/2.html)

이 노트북은 책 2.2절의 내용을 코드로 재현합니다:

1. **UCB 보너스** `c·sqrt(ln t / N)`를 본문 "손으로 한 번"의 숫자 그대로 확인.
2. **보너스 감소 곡선**(1/√N) + UCB 점수가 팔 선택을 뒤집는 그림.
3. **세 전략 대결**(ε-greedy vs 낙관적 초기화 vs UCB) — 누적 보상과 팔 당김 횟수.
4. **팔 개수 k=10**으로 늘렸을 때 탐색 비용의 차이.
5. **비정상 환경**(중간에 팔 순위가 뒤집힘)에서 누가 새 최선에 적응하는지.

numpy/matplotlib만 씁니다 — 외부 데이터 다운로드 불필요, CPU만으로 충분.
모든 실행은 `random.seed(0)`로 고정되어 재현됩니다.

## 0. 환경 준비

그래프의 한글 라벨이 깨지지 않도록 CJK 폰트를 골라둡니다 (Colab에 기본 설치).
그림은 `kor/src/images/`로 저장합니다 (Colab 등에서는 `/tmp`로 자동 대체).

In [1]:
import math
import random

import numpy as np
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.sans-serif"] = ["Noto Sans CJK KR", "NanumGothic", "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt

import os
IMG = "/home/smhan/book-ml/kor/src/images"
if not os.path.isdir(IMG):   # Colab 등에서는 /tmp로 자동 대체
    IMG = "/tmp"
SEED = 0
STEPS = 2000
print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print(f"그림 저장 위치: {IMG}   (SEED={SEED}, STEPS={STEPS})")

numpy 2.4.6 | matplotlib 3.11.1
그림 저장 위치: /home/smhan/book-ml/kor/src/images   (SEED=0, STEPS=2000)


## 1. UCB 보너스: 본문의 "손으로 한 번" 숫자 그대로 확인

본문의 예: \\(t=100, c=2\\)에서 \\(N=5\\) 팔의 보너스는
\\(2\sqrt{\ln 100 / 5} \approx 1.919\\), \\(N=50\\) 팔은
\\(\approx 0.607\\). 그리고 \\(t=100\\), \\(c=2\\) 시점의 세 팔
(A: \\(Q=2.50, N=50\\) / B: \\(Q=2.00, N=20\\) / C: \\(Q=1.60, N=5\\))의
UCB 점수는 A=3.107, B=2.960, **C=3.519** — 추정치 최하 C가 1위가 된다.
이 숫자를 코드로 계산해 맞혀봅니다.

In [2]:
t, c = 100, 2.0
print(f"t={t}, c={c}   (ln t = {math.log(t):.3f})")
for N in (5, 20, 50):
    b = c * math.sqrt(math.log(t) / N)
    print(f"  N={N:>2}: 보너스 = {b:.3f}")
assert abs(c * math.sqrt(math.log(t) / 5)  - 1.919) < 1e-3
assert abs(c * math.sqrt(math.log(t) / 20) - 0.960) < 1e-3
assert abs(c * math.sqrt(math.log(t) / 50) - 0.607) < 1e-3
print("[OK] 보너스 값이 본문의 손계산(1.919 / 0.960 / 0.607)과 일치")

arms = [("A", 2.50, 50), ("B", 2.00, 20), ("C", 1.60, 5)]
print("\n팔  Q_t    N    보너스    UCB 점수")
scores = {}
for name, q, N in arms:
    b = c * math.sqrt(math.log(t) / N)
    scores[name] = q + b
    print(f"{name}   {q:4.2f}   {N:>2}   {b:6.3f}    {q + b:6.3f}")
assert scores["C"] > scores["A"] > scores["B"]
print(f"[OK] UCB 점수 순서 C({scores['C']:.3f}) > A({scores['A']:.3f}) > B({scores['B']:.3f})")
print("     -> 순수 탐욕의 1위(A)를 '덜 당겨본' 팔 C가 보너스로 앞선다")

t=100, c=2.0   (ln t = 4.605)
  N= 5: 보너스 = 1.919
  N=20: 보너스 = 0.960
  N=50: 보너스 = 0.607
[OK] 보너스 값이 본문의 손계산(1.919 / 0.960 / 0.607)과 일치

팔  Q_t    N    보너스    UCB 점수
A   2.50   50    0.607     3.107
B   2.00   20    0.960     2.960
C   1.60    5    1.919     3.519
[OK] UCB 점수 순서 C(3.519) > A(3.107) > B(2.960)
     -> 순수 탐욕의 1위(A)를 '덜 당겨본' 팔 C가 보너스로 앞선다


## 2. 그림 1: 보너스의 1/√N 감소 + UCB 점수

(왼쪽) \\(N\\)이 커질수록 보너스가 \\(1/\sqrt{N}\\)으로 감소하는 곡선.
N이 4배가 되면 보너스는 정확히 절반으로 준다 — "√n의 법칙".
(오른쪽) 1번 섹션의 세 팔 UCB 점수: **막대 아래(Q_t)는 증거,
위(보너스)는 불확실성**. C가 증거에선 최하인데 불확실성 보너스 덕분에
합계 1위가 된다.

In [3]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.3))

# (왼쪽) 보너스 감소 곡선
Ns = np.arange(1, 201)
bonuses = c * np.sqrt(np.log(t) / Ns)
ax1.plot(Ns, bonuses, lw=2, color="#1d4ed8")
for N, col in [(5, "#dc2626"), (20, "#f59e0b"), (50, "#059669")]:
    b = c * math.sqrt(math.log(t) / N)
    ax1.plot([N], [b], "o", ms=7, color=col, zorder=5)
    ax1.annotate(f"N={N}: {b:.2f}", (N, b), textcoords="offset points",
                 xytext=(8, -14), fontsize=9, color=col)
ax1.set_xlabel("Number of pulls N for arm a")
ax1.set_ylabel(f"Bonus c·√(ln t / N)  (t={t}, c={c:.0f})")
ax1.set_title("UCB bonus: 1/√N decay (4×N → half the bonus)")
ax1.grid(alpha=0.3)

# (오른쪽) 세 팔의 UCB 점수 (stacked bar: Q + 보너스)
names = [a[0] for a in arms]
qs = np.array([a[1] for a in arms])
bs = np.array([c * math.sqrt(math.log(t) / a[2]) for a in arms])
x = np.arange(3)
ax2.bar(x, qs, color="#93c5fd", label="Q_t(a) (evidence)")
ax2.bar(x, bs, bottom=qs, color="#fca5a5", label="Bonus c·√(ln t/N)")
for i, n in enumerate(names):
    ax2.text(x[i], qs[i] + bs[i] + 0.05, f"{qs[i]+bs[i]:.2f}",
             ha="center", fontsize=10, fontweight="bold")
ax2.set_xticks(x, names)
ax2.set_ylabel("UCB score")
ax2.set_title("UCB score: evidence (Q_t) + uncertainty bonus")
ax2.legend(fontsize=9)
ax2.grid(alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(f"{IMG}/ch02_2_ucb_bonus.svg")
plt.show()
print("저장:", f"{IMG}/ch02_2_ucb_bonus.svg")

저장: /home/smhan/book-ml/kor/src/images/ch02_2_ucb_bonus.svg


## 3. 세 전략 구현

본문에 있는 `total_reward_*` 함수를 그대로 쓰고(2.2절 실습 코드),
그림을 그리려면 **누적 보상**과 **당김 횟수**까지 기록하는
`run_strategy`을 추가로 정의합니다 — 선택 로직은 정확히 같습니다.
공정한 비교를 위해 **각 전략마다 seed를 다시 걸고** 독립 실행합니다.

In [4]:
def total_reward_epsilon_greedy(true_means, epsilon, steps):
    k = len(true_means)
    Q, N, total = [0.0] * k, [0] * k, 0.0
    for _ in range(steps):
        a = random.randrange(k) if random.random() < epsilon \
            else max(range(k), key=lambda i: Q[i])
        r = random.gauss(true_means[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        total += r
    return total

def total_reward_optimistic(true_means, initial_q, steps):
    k = len(true_means)
    Q, N, total = [initial_q] * k, [0] * k, 0.0  # 실제 보상보다 훨씬 큰 초기값
    for _ in range(steps):
        a = max(range(k), key=lambda i: Q[i])  # 탐욕적 선택뿐이지만
        r = random.gauss(true_means[a], 1.0)   # 낙관적 초기값 덕에 자동 탐험됨
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        total += r
    return total

def total_reward_ucb(true_means, c, steps):
    k = len(true_means)
    Q, N, total = [0.0] * k, [0] * k, 0.0
    for t in range(1, steps + 1):
        unplayed = [i for i in range(k) if N[i] == 0]
        a = unplayed[0] if unplayed else \
            max(range(k), key=lambda i: Q[i] + c * math.sqrt(math.log(t) / N[i]))
        r = random.gauss(true_means[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        total += r
    return total

def run_strategy(true_means, strategy, steps, param):
    """strategy: 'eps' | 'opt' | 'ucb' ; param: ε | 초기값 | c.
    (total, Q, N, cum) 반환 — cum[i]는 i+1스텝까지의 누적 보상."""
    k = len(true_means)
    Q = [param] * k if strategy == "opt" else [0.0] * k
    N = [0] * k
    total, cum = 0.0, []
    for t in range(1, steps + 1):
        if strategy == "eps":
            a = random.randrange(k) if random.random() < param else max(range(k), key=lambda i: Q[i])
        elif strategy == "opt":
            a = max(range(k), key=lambda i: Q[i])
        else:
            unplayed = [i for i in range(k) if N[i] == 0]
            if unplayed:
                a = unplayed[0]
            else:
                a = max(range(k), key=lambda i: Q[i] + param * math.sqrt(math.log(t) / N[i]))
        r = random.gauss(true_means[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        total += r
        cum.append(total)
    return total, Q, N, cum

## 4. 세 전략 대결: seed=0, k=3, 2000스텝

본문 표의 숫자(ε-greedy 3893.9 / 낙관적 초기화 4007.7 / UCB 3940.2)를
확인합니다. **각 전략마다 `random.seed(0)`을 다시** 걸어 같은 무작위
시퀀스로 독립 실행하는 것이 본문의 실행 방식과 같습니다.

In [5]:
true_means = [1.0, 1.5, 2.0]
random.seed(0)
tot_eps = total_reward_epsilon_greedy(true_means, 0.1, STEPS)
random.seed(0)
tot_opt = total_reward_optimistic(true_means, 5.0, STEPS)
random.seed(0)
tot_ucb = total_reward_ucb(true_means, 2.0, STEPS)
print(f"ε-greedy 총 보상:    {tot_eps:.1f}")
print(f"낙관적 초기화 총 보상: {tot_opt:.1f}")
print(f"UCB 총 보상:         {tot_ucb:.1f}")
assert abs(tot_eps - 3893.9) < 0.05
assert abs(tot_opt - 4007.7) < 0.05
assert abs(tot_ucb - 3940.2) < 0.05
print("[OK] 본문 '실습에서 실제로 나오는 숫자' 표와 일치")

# 그림용: 누적 보상 + 당김 횟수 기록 (같은 seed, 같은 로직)
random.seed(0); tot_e, Qe, Ne, cume = run_strategy(true_means, "eps", STEPS, 0.1)
random.seed(0); tot_o, Qo, No, cumo = run_strategy(true_means, "opt", STEPS, 5.0)
random.seed(0); tot_u, Qu, Nu, cumu = run_strategy(true_means, "ucb", STEPS, 2.0)
print(f"\n당김 횟수 N = (A, B, C):")
print(f"  ε-greedy      : {Ne}   (나쁜 팔 A,B를 계속 다시 당김)")
print(f"  낙관적 초기화  : {No}   (각 팔 최소 1회 후 최선 팔에 집중)")
print(f"  UCB           : {Nu}")
assert Ne == [82, 50, 1868] and No == [3, 1, 1996] and Nu == [35, 72, 1893]
print("[OK] 당김 횟수까지 본문 표와 정확히 일치")

ε-greedy 총 보상:    3893.9
낙관적 초기화 총 보상: 4007.7
UCB 총 보상:         3940.2
[OK] 본문 '실습에서 실제로 나오는 숫자' 표와 일치

당김 횟수 N = (A, B, C):
  ε-greedy      : [82, 50, 1868]   (나쁜 팔 A,B를 계속 다시 당김)
  낙관적 초기화  : [3, 1, 1996]   (각 팔 최소 1회 후 최선 팔에 집중)
  UCB           : [35, 72, 1893]
[OK] 당김 횟수까지 본문 표와 정확히 일치


## 5. 그림 2: 누적 보상 곡선 + 팔 당김 횟수

(위) 세 전략의 누적 보상 — 초반(첫 ~10스텝)은 셋 다 비슷하지만,
중반부터 "탐험의 낭비"가 차이로 벌어진다.
(아래) 각 팔의 최종 당김 횟수 — ε-greedy만 나쁜 팔 A·B를 대량으로
다시 당긴 것이 한눈에 보인다.

In [6]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7.5),
                               gridspec_kw={"height_ratios": [1.4, 1]})
steps_x = np.arange(1, STEPS + 1)
ax1.plot(steps_x, cume, lw=1.5, label=f"ε-greedy (ε=0.1)  → {tot_e:.0f}", color="#888888")
ax1.plot(steps_x, cumo, lw=1.8, label=f"Optimistic init (Q₀=5.0)  → {tot_o:.0f}", color="#ff7f0e")
ax1.plot(steps_x, cumu, lw=1.8, label=f"UCB (c=2)  → {tot_u:.0f}", color="#1f77b4")
ax1.set_xlabel("Step")
ax1.set_ylabel("Cumulative reward")
ax1.set_title("Cumulative reward of the three strategies (true means [1.0, 1.5, 2.0], seed=0, 2000 steps)")
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

arms_name = ["A (q*=1.0)", "B (q*=1.5)", "C (q*=2.0)"]
strategies = [("ε-greedy", Ne, "#888888"), ("Optimistic init", No, "#ff7f0e"), ("UCB", Nu, "#1f77b4")]
left = np.zeros(3)
x = np.arange(len(strategies))
for sname, Ns, col in strategies:
    ax2.bar(x, Ns, bottom=left, color=col, label=sname, alpha=0.9)
    for i in range(3):
        if Ns[i] > 60:
            ax2.text(x[0] if False else x[list(strategies).index((sname, Ns, col))],
                     left[i] + Ns[i]/2, f"{Ns[i]}", ha="center", va="center",
                     fontsize=8, color="white")
    left += np.array(Ns)
ax2.set_xticks(x, [s[0] for s in strategies])
ax2.set_ylabel("Pulls")
ax2.set_title("Final pulls per arm (color = arm)")
ax2.legend(fontsize=9, loc="upper left")
ax2.grid(alpha=0.3, axis="y")

fig.tight_layout()
fig.savefig(f"{IMG}/ch02_2_ucb_h2h.svg")
plt.show()
print("저장:", f"{IMG}/ch02_2_ucb_h2h.svg")

저장: /home/smhan/book-ml/kor/src/images/ch02_2_ucb_h2h.svg


/tmp/ipykernel_832067/3426771105.py:31: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Noto Sans CJK KR.
  fig.tight_layout()
/tmp/ipykernel_832067/3426771105.py:32: UserWarning: Glyph 8320 (\N{SUBSCRIPT ZERO}) missing from font(s) Noto Sans CJK KR.
  fig.savefig(f"{IMG}/ch02_2_ucb_h2h.svg")


## 6. 팔 개수가 늘어나면: k=10

평균이 0.5~1.3 (0.1 간격)인 9개 팔 + 진짜 최선 1개(2.0). 같은 seed=0,
2000스텝. **낙관적 초기화는 각 팔을 정확히 1회만** 확인하는 최소
비용으로 최선에 집중하고, **UCB는 9개 팔에 합계 214회**를 써서
"불확실한 곳"만 골라 재확인한다(서브선형 억제).

In [7]:
means10 = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.1, 1.2, 1.3, 2.0]
random.seed(0)
tot10_o, Q10o, N10o, _ = run_strategy(means10, "opt", STEPS, 5.0)
random.seed(0)
tot10_u, Q10u, N10u, _ = run_strategy(means10, "ucb", STEPS, 2.0)
print(f"낙관적 초기화: 총 보상 {tot10_o:.1f}   N = {N10o}")
print(f"UCB:         총 보상 {tot10_u:.1f}   N = {N10u}")
print(f"  -> 낙관적 초기화: 9개 나쁜 팔을 '각 1회'만 (탐색 비용 = k-1 = 9회)")
print(f"  -> UCB: 9개 나쁜 팔 합계 {sum(N10u[:9])}회 (각 10~53회, 서브선형 억제)")
assert abs(tot10_o - 4001.3) < 0.05 and N10o == [1]*9 + [1991]
assert abs(tot10_u - 3802.1) < 0.05 and N10u == [12, 18, 11, 15, 10, 24, 47, 24, 53, 1786]
print("[OK] 본문 '팔 개수가 늘어나면' 예의 숫자와 일치")

낙관적 초기화: 총 보상 4001.3   N = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1991]
UCB:         총 보상 3802.1   N = [12, 18, 11, 15, 10, 24, 47, 24, 53, 1786]
  -> 낙관적 초기화: 9개 나쁜 팔을 '각 1회'만 (탐색 비용 = k-1 = 9회)
  -> UCB: 9개 나쁜 팔 합계 214회 (각 10~53회, 서브선형 억제)
[OK] 본문 '팔 개수가 늘어나면' 예의 숫자와 일치


## 7. 비정상 환경: 1500스텝에서 팔 순위가 뒤집힌다

처음 1500스텝: \\(q^* = [1.0, 1.5, 2.0]\\) (**C가 최선**).
그 뒤 500스텝: \\(q^* = [0.5, 2.0, 1.0]\\)으로 순위가 뒤집혀
(**B가 새 최선**, C는 중간으로 하락). "환경이 바뀌었는지 감지하고
적응"하는 전략이 누구인지, **마지막 500스텝의 평균 보상**으로
판정합니다 (새 최선 B의 진짜 평균 = 2.0).

In [8]:
before, after = [1.0, 1.5, 2.0], [0.5, 2.0, 1.0]
CHANGE_AT = 1500

def run_ns(strategy, param, change_at=CHANGE_AT):
    k = 3
    Q = [param] * k if strategy == "opt" else [0.0] * k
    N = [0] * k
    rewards = []
    for t in range(1, STEPS + 1):
        means = after if t >= change_at else before
        if strategy == "eps":
            a = random.randrange(k) if random.random() < param else max(range(k), key=lambda i: Q[i])
        elif strategy == "opt":
            a = max(range(k), key=lambda i: Q[i])
        else:
            unplayed = [i for i in range(k) if N[i] == 0]
            a = unplayed[0] if unplayed else \
                max(range(k), key=lambda i: Q[i] + param * math.sqrt(math.log(t) / N[i]))
        r = random.gauss(means[a], 1.0)
        N[a] += 1
        Q[a] += (r - Q[a]) / N[a]
        rewards.append(r)
    last500 = sum(rewards[-500:]) / 500.0
    return last500, Q, N, rewards

random.seed(0); avg_e, Qe, Ne, rew_e = run_ns("eps", 0.1)
random.seed(0); avg_o, Qo, No, rew_o = run_ns("opt", 5.0)
random.seed(0); avg_u, Qu, Nu, rew_u = run_ns("ucb", 2.0)
print("전략                 마지막500스텝평균   마지막Q           마지막N")
for nm, av, Q, N in [("ε-greedy (ε=0.1)", avg_e, Qe, Ne),
                     ("낙관적 초기화 (Q₀=5.0)", avg_o, Qo, No),
                     ("UCB (c=2)", avg_u, Qu, Nu)]:
    print(f"  {nm:22s}  {av:+.2f}        " +
          "  ".join(f"{q:.2f}" for q in Q) + f"    {N}")
assert abs(avg_e - 0.97) < 0.01 and abs(avg_o - 1.02) < 0.01 and abs(avg_u - 1.86) < 0.01
assert Nu == [34, 492, 1474]
print("[OK] 본문 '비정상 환경' 표와 일치 — UCB만 새 최선(B, 평균 2.0)에 근접")
print(f"     UCB: 새 최선 B를 {Nu[1]}회 재확인 (Q_B = {Qu[1]:.2f}) | "
      f"낙관적 초기화: B를 {No[1]}회만 (Q_B = {Qo[1]:.2f}으로 완전히 틀림)")

전략                 마지막500스텝평균   마지막Q           마지막N
  ε-greedy (ε=0.1)        +0.97        0.89  1.41  1.75    [82, 50, 1868]
  낙관적 초기화 (Q₀=5.0)        +1.02        1.10  0.10  1.76    [3, 1, 1996]
  UCB (c=2)               +1.86        1.13  1.91  1.96    [34, 492, 1474]
[OK] 본문 '비정상 환경' 표와 일치 — UCB만 새 최선(B, 평균 2.0)에 근접
     UCB: 새 최선 B를 492회 재확인 (Q_B = 1.91) | 낙관적 초기화: B를 1회만 (Q_B = 0.10으로 완전히 틀림)


## 8. 그림 3: 환경 변화(1500스텝) 이후의 적응 속도

50스텝 이동 평균 보상. 1500스텝에서 환경이 바뀌면, **UCB만**
새 최선(B, 평균 2.0) 수준으로 회복하고, 낙관적 초기화는 예전 최선이던
C에 매달려 ~1.0에 갇힌다 — "탐험을 매 스텝 재계산"이 환경 변화를
감지하는 메커니즘임을 보인다.

In [9]:
def moving_avg(rewards, w=50):
    out = []
    for i in range(len(rewards)):
        j0 = max(0, i - w + 1)
        out.append(sum(rewards[j0:i+1]) / (i - j0 + 1))
    return np.array(out)

sx = np.arange(1, STEPS + 1)
fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(sx, moving_avg(rew_e), lw=1.3, color="#888888", label=f"ε-greedy (last-500-step avg {avg_e:+.2f})")
ax.plot(sx, moving_avg(rew_o), lw=1.5, color="#ff7f0e", label=f"Optimistic init ({avg_o:+.2f})")
ax.plot(sx, moving_avg(rew_u), lw=1.8, color="#1f77b4", label=f"UCB ({avg_u:+.2f})")
ax.axvline(CHANGE_AT, color="k", ls="--", lw=1)
ax.text(CHANGE_AT + 15, ax.get_ylim()[0] + 0.05, "Environment change (ranking reversal)", fontsize=9)
ax.axhline(2.0, color="#1f77b4", ls=":", lw=1, alpha=0.6)
ax.text(1010, 2.02, "Mean 2.0 of new best arm B", fontsize=8, color="#1f77b4")
ax.set_xlabel("Step")
ax.set_ylabel("Reward (50-step moving average)")
ax.set_title("Non-stationary environment: q* = [1.0,1.5,2.0] → [0.5,2.0,1.0] at step 1500")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 9. 정리

| 확인한 것 | 결론 |
|---|---|
| 보너스 손계산 (t=100, c=2) | N=5→**1.919**, N=20→0.960, N=50→**0.607** — 코드 일치 |
| UCB 점수 | 추정치 최하 C(1.60)가 보너스 1.92로 1위(3.52) — 탐험이 활용을 이김 |
| 3팔 대결 (seed=0) | 낙관적 초기화 **4007.7** > UCB 3940.2 > ε-greedy 3893.9 |
| k=10 | 낙관적 초기화: 각 팔 **정확히 1회**(비용 k−1); UCB: 9팔 합계 214회(서브선형) |
| 비정상 환경 (순위 뒤집힘) | UCB **+1.86**(새 최선에 근접) / 낙관적 초기화 +1.02(예전 최선에 고착) |

세 전략은 같은 "탐험"을 세 가지 방식으로 만든다 —
**(a) ε-greedy: 고정 확률로, (b) 낙관적 초기화: 초기값 하나로 예약,
(c) UCB: 매 스텝의 불확실성으로 재계산**. 고정 환경에서는 (b)가
가장 가볍고 효율적이지만, **환경이 바뀌는 문제에서는 (c)의
"계속 재계산"만이 적응**한다. 이 "불확실성에 비례하는 탐험" 원리는
Chapter 6(TD 학습)의 ε-greedy, 그리고 그 뒤의 Q-학습·DQN까지
그대로 이어진다.